# DeBERTa-v3-base v2

Melhorias face ao v1:
- **Mais trials Optuna** (15 em vez de 10)
- **Range de LR mais alto** (1e-5 a 5e-5, como o RoBERTa que funcionou melhor)
- **MAX_EPOCHS nas trials** aumentado para 10 (v1 usava 6, pouco para DeBERTa convergir)
- **Patience mais alto** nas trials (3 em vez de 2)
- **Dados Subm1 + Subm2** como dados reais
- **GradScaler** para mixed precision (como o RoBERTa)

In [1]:
import os
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DebertaV2Tokenizer, DebertaV2ForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformer_utils import TextDataset, label_smoothing_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID  = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
N_CLASSES = 5

device: cuda


## *dados*

In [2]:
df_full  = pd.read_csv('../datasets/dataset_v2_full.csv',    sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',   sep=';')
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv', sep=';')
df_subm2 = pd.read_csv('../Subm2/subm2_labels_revealed.csv', sep=';')

def load_xy(df):
    df = df.dropna(subset=['Label'])
    df = df[df['Label'].str.strip().str.lower().isin(LABEL2ID.keys())]
    texts  = df['Text'].fillna('').tolist()
    labels = [LABEL2ID[l.strip().lower()] for l in df['Label'].tolist()]
    return texts, labels

texts_synth, y_synth = load_xy(df_full)
texts_r1,    y_r1    = load_xy(df_subm1)
texts_r2,    y_r2    = load_xy(df_subm2)
texts_real   = texts_r1 + texts_r2
y_real       = y_r1     + y_r2
texts_val,   y_val   = load_xy(df_ex)

# class weights baseados nos dados reais
cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_real + y_val)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

print(f'sint\u00e9ticos: {len(texts_synth)} | reais: {len(texts_real)} (subm1: {len(texts_r1)} + subm2: {len(texts_r2)})')
print(f'valida\u00e7\u00e3o:  {len(texts_val)} exemplos do docente')
print('class weights:', {ID2LABEL[i]: f'{w:.2f}' for i, w in enumerate(cw)})

sintéticos: 5000 | reais: 200 (subm1: 100 + subm2: 100)
validação:  125 exemplos do docente
class weights: {'google': '1.27', 'anthropic': '1.16', 'meta': '1.27', 'openai': '1.38', 'human': '0.54'}


## *modelo e funções auxiliares*

In [3]:
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN    = 128

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)


def evaluate_model(model, loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds_all.extend(outputs.logits.argmax(dim=1).cpu().tolist())
            labels_all.extend(batch['labels'].tolist())
    acc = sum(p == t for p, t in zip(preds_all, labels_all)) / len(labels_all)
    f1  = f1_score(labels_all, preds_all, average='macro')
    return acc, f1, preds_all

print(f'modelo base: {MODEL_NAME}')

Could not extract SentencePiece model from C:\Users\gbarr\.cache\huggingface\hub\models--microsoft--deberta-v3-base\snapshots\8ccc9b6f36199bec6961081d44eb72fb3f7353f3\spm.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in C:\Users\gbarr\.cache\huggingface\hub\models--microsoft--deberta-v3-base\snapshots\8ccc9b6f36199bec6961081d44eb72fb3f7353f3\spm.model

## *Optuna — otimização de hiperparâmetros*

Diferenças chave vs v1:
- LR range 1e-5 a 5e-5 (v1 usava 5e-6 a 3e-5, demasiado baixo)
- 10 epochs por trial (v1 usava 6)
- Patience 3 (v1 usava 2)
- GradScaler para mixed precision estável

In [ ]:
import optuna

val_ds     = TextDataset(texts_val, y_val, tokenizer, MAX_LEN)
val_loader = DataLoader(val_ds, batch_size=32)
print(f'valida\u00e7\u00e3o tokenizada: {len(val_ds)} amostras')


def train_one_trial(trial):
    lr           = trial.suggest_float('lr', 1e-5, 5e-5, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [16, 32])
    smoothing    = trial.suggest_float('label_smoothing', 0.05, 0.2)
    warmup_frac  = trial.suggest_float('warmup_frac', 0.05, 0.2)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 0.1, log=True)
    real_wt      = trial.suggest_int('real_weight', 5, 20)

    MAX_EPOCHS = 10
    PATIENCE   = 3

    t_train = texts_synth + texts_real * real_wt
    y_tr    = y_synth     + y_real     * real_wt
    train_ds     = TextDataset(t_train, y_tr, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DebertaV2ForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=ID2LABEL, label2id=LABEL2ID
    ).float().to(device)

    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps  = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(total_steps * warmup_frac)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

    best_f1    = 0.0
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                              smoothing=smoothing, weights=class_weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        val_acc, val_f1, _ = evaluate_model(model, val_loader)

        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_f1 > best_f1:
            best_f1    = val_f1
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    del model, optimizer, scheduler, scaler, train_ds, train_loader
    torch.cuda.empty_cache()

    return best_f1


print('fun\u00e7\u00e3o de treino definida!')

In [ ]:
N_TRIALS = 15

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
    study_name='deberta-v2-hparam-search'
)
study.optimize(train_one_trial, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nmelhor F1-macro: {study.best_value:.4f}')
print('melhores hiperpar\u00e2metros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## *treino final*

In [ ]:
bp = study.best_params
LR           = bp['lr']
BATCH_SIZE   = bp['batch_size']
LABEL_SMOOTH = bp['label_smoothing']
WARMUP_FRAC  = bp['warmup_frac']
WEIGHT_DECAY = bp['weight_decay']
REAL_WT      = bp['real_weight']
MAX_EPOCHS   = 20
PATIENCE     = 5

t_train_final = texts_synth + texts_real * REAL_WT
y_train_final = y_synth     + y_real     * REAL_WT

train_ds     = TextDataset(t_train_final, y_train_final, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f'treino final: {len(train_ds)} amostras | batch_size={BATCH_SIZE}')
print(f'lr={LR:.2e} | smoothing={LABEL_SMOOTH:.4f} | warmup={WARMUP_FRAC:.4f} | wd={WEIGHT_DECAY:.4f} | real_wt={REAL_WT}')

In [ ]:
model = DebertaV2ForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID
).float().to(device)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

best_f1    = 0.0
best_acc   = 0.0
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss  = 0.0
    running_total = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                          smoothing=LABEL_SMOOTH, weights=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss  += loss.item() * input_ids.size(0)
        running_total += input_ids.size(0)

    train_loss          = running_loss / running_total
    val_acc, val_f1, _ = evaluate_model(model, val_loader)

    marker = ' *' if val_f1 > best_f1 else ''
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_acc   = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'early stopping na \u00e9poca {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nmelhor modelo -> val_acc={best_acc:.4f} | val_f1={best_f1:.4f}')

In [ ]:
SAVE_DIR = '../models/model_deberta'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'modelo guardado em {SAVE_DIR}')

## *avaliação com dataset-exemplos*

In [ ]:
val_acc, val_f1, val_preds = evaluate_model(model, val_loader)
print(f'[dataset-exemplos] accuracy={val_acc:.4f} | f1-macro={val_f1:.4f}')
print()
print(classification_report(
    [ID2LABEL[l] for l in y_val],
    [ID2LABEL[p] for p in val_preds],
    digits=3
))

## *resultados Optuna*

In [ ]:
trials_df = study.trials_dataframe().sort_values('value', ascending=False)

cols = ['number', 'value', 'params_lr', 'params_batch_size',
        'params_label_smoothing', 'params_warmup_frac',
        'params_weight_decay', 'params_real_weight']
display_cols = [c for c in cols if c in trials_df.columns]

print('top 5 trials:')
print(trials_df[display_cols].head(5).to_string(index=False))

## *próximo passo*

Se o F1 subiu para ~65%+, voltar a correr o `train_ensemble.ipynb` para ver se o stacking agora melhora.